<a href="https://colab.research.google.com/github/shubhampuri240/manisha-overseas-website/blob/staging/stage4/Manisha_Overseas_Free_TryOn_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Manisha Overseas · Stage 4 — Free AI Try-On Test (Google Colab)

**Research / internal evaluation only. Do not deploy this as a customer-facing website feature.** This notebook tests the upstream FASHN VTON v1.5 model using the **upstream sample person image** and one of Manisha Overseas’ approved garment photographs. It does **not** request personal photographs, open a web server, publish any images to GitHub, or connect to commercial FASHN services.

### Important before running
- **Pricing:** Colab's free tier can offer GPUs, but GPU availability and usage limits are not guaranteed.
- **Licensing:** The FASHN VTON code and main weights are Apache-2.0. The pipeline depends on `fashn-human-parser`, whose SegFormer-based model inherits NVIDIA's **non-commercial research/evaluation-only** terms. This notebook is a *restricted internal test*. Commercial launch needs permission or a licensed replacement.
- **Colab policy:** Colab prohibits using managed runtimes to create deepfakes. Use only the upstream supplied demo person image in this notebook; do not generate altered images of actual customers or other people here.
- **Privacy:** Images stay in the temporary Colab runtime, but Google processes the runtime and may retain service data. The garment is fetched from your public GitHub staging branch. Do not mount Google Drive or upload private/customer images.
- **GPU:** This model needs substantial GPU memory (~8 GB in bfloat16 on supported GPUs). A free older T4 GPU can use a less memory-efficient precision and may run out of memory.

Official model: https://github.com/fashn-AI/fashn-vton-1.5  
Google Colab usage policy: https://research.google.com/colaboratory/faq.html

**Start by choosing: Runtime → Change runtime type → T4 GPU (or another available GPU) → Save.** Run the cells one at a time in order.

## 1 · Verify free GPU

This stops with a useful message if the runtime has no GPU. If you cannot choose a free GPU, you do not need to purchase anything; retry only when a free GPU is offered.

In [ ]:
import torch
from IPython.display import display, Markdown

if not torch.cuda.is_available():
    raise RuntimeError("No GPU detected. In Colab: Runtime → Change runtime type → GPU. Free GPU availability is not guaranteed.")

gpu = torch.cuda.get_device_properties(0)
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM: %.1f GB" % (gpu.total_memory / (1024**3)))
print("CUDA capability:", f"{gpu.major}.{gpu.minor}")
if gpu.major < 8:
    print("WARNING: This older GPU may need float32 and run out of VRAM. The notebook is not guaranteed to work on a free T4.")
display(Markdown("✅ GPU runtime detected. **No payment is required to run on the free tier if one is allocated.**"))

## 2 · Install the official FASHN VTON repository

This downloads and installs open-source model code directly from the official repository, rather than uploading any secret keys.

In [ ]:
import os, sys, subprocess
from pathlib import Path

ROOT = Path("/content")
REPO = ROOT / "fashn-vton-1.5"
if not (REPO / "pyproject.toml").is_file():
    subprocess.run(
        ["git", "clone", "--depth", "1", "https://github.com/fashn-AI/fashn-vton-1.5.git", str(REPO)],
        check=True
    )
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO)], check=True)
print("Official model package installed. If an import fails later after dependency changes, restart the runtime and rerun this notebook.")

## 3 · Download the model weights

The official download script obtains approximately **2 GB** of model and pose weights. Human-parser weights may download on first generation.

In [ ]:
WEIGHTS = REPO / "weights"
subprocess.run(
    [sys.executable, str(REPO / "scripts" / "download_weights.py"), "--weights-dir", str(WEIGHTS)],
    check=True
)
print("Weights ready:", WEIGHTS)

## 4 · Select an actual Manisha Overseas garment

Change `STYLE_CODE` to any code in the list. This fetches an **original catalogue image from GitHub's staging branch**; no photograph is recompressed or edited on GitHub. The model will resize its input internally.

In [ ]:
from io import BytesIO
import requests
from PIL import Image, ImageOps
from IPython.display import display

STYLES = {
    "4638": ("Floral printed long dress", "style-4638-front.webp"),
    "6821": ("Printed wide-leg one-piece", "style-6821-front.webp"),
    "3476": ("Printed long dress", "style-3476-front.webp"),
    "7284": ("Open-back long dress", "style-7284-front.webp"),
    "1509": ("Tiered long dress", "style-1509-front.webp"),
    "2941": ("Printed tiered dress", "style-2941-front.webp"),
    "9152": ("Green printed long dress", "style-9152-full.webp"),
}
STYLE_CODE = "4638"  # Change this to "3476", "9152", etc.

if STYLE_CODE not in STYLES:
    raise ValueError(f"Unknown style {STYLE_CODE}. Choose from: {', '.join(STYLES)}")
label, filename = STYLES[STYLE_CODE]
url = (
    "https://raw.githubusercontent.com/"
    "shubhampuri240/manisha-overseas-website/staging/assets/visuals/" + filename
)
with requests.get(url, timeout=45, stream=True) as response:
    response.raise_for_status()
    data = bytearray()
    for chunk in response.iter_content(chunk_size=65536):
        data.extend(chunk)
        if len(data) > 15 * 1024 * 1024:
            raise ValueError("Garment file larger than the 15 MB evaluation limit.")
with Image.open(BytesIO(data)) as im:
    if im.width * im.height > 30_000_000:
        raise ValueError("Garment photograph exceeds 30 megapixels.")
    garment = ImageOps.exif_transpose(im).convert("RGB").copy()
print(f"Selected Manisha Overseas style {STYLE_CODE}: {label}")
display(garment.resize((int(garment.width * min(420/garment.width, 1)), int(garment.height * min(420/garment.width, 1)))))

## 5 · Load the model's public demo person

**No upload required.** Using the model's bundled sample avoids private customer photos and tests image quality before any further integration work.

In [ ]:
from PIL import Image
demo_path = REPO / "examples" / "data" / "model.webp"
if not demo_path.exists():
    raise FileNotFoundError("Official example image not found: " + str(demo_path))
with Image.open(demo_path) as im:
    person = im.convert("RGB").copy()
print("Using official sample person, not a customer photo.")
display(person.resize((int(person.width * min(420/person.width, 1)), int(person.height * min(420/person.width, 1)))))

In [ ]:
import os
import sys
import subprocess

repo = "/content/fashn-vton-1.5"

if not os.path.isdir(repo):
    subprocess.check_call([
        "git", "clone",
        "https://github.com/fashn-AI/fashn-vton-1.5.git",
        repo
    ])

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-e", repo
])

from fashn_vton import TryOnPipeline
print("FASHN VTON installed successfully!")


## 6 · Load and run AI virtual try-on

This is the real FASHN VTON inference model, **not a 3D garment model or a clothing overlay**. GPU and dependency compatibility can vary in free Colab. If you see an out-of-memory error, do not pay; report the GPU name and error so we can choose a smaller alternative.

In [13]:
import sys
from pathlib import Path
import torch

# Set core directories
ROOT = Path("/content")
WEIGHTS = ROOT / "fashn-vton-1.5" / "weights"

# Ensure both repository root and src directory are in Python path to guarantee resolution
REPO_ROOT = ROOT / "fashn-vton-1.5"
REPO_SRC = REPO_ROOT / "src"

for path in [REPO_ROOT, REPO_SRC]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from fashn_vton import TryOnPipeline
import gc

if not torch.cuda.is_available():
    raise RuntimeError("The GPU became unavailable. Reconnect a GPU runtime before trying again.")
torch.cuda.empty_cache()
pipeline = TryOnPipeline(weights_dir=str(WEIGHTS), device="cuda")
print("Model loaded. Generating one internal test image...")
try:
    output = pipeline(
        person_image=person,
        garment_image=garment,
        category="one-pieces",
        garment_photo_type="model",
        num_samples=1,
        num_timesteps=20,
        seed=42,
        segmentation_free=True,
    )
except torch.cuda.OutOfMemoryError:
    raise RuntimeError(
        "GPU ran out of memory. Free Colab may assign an older GPU such as T4. "
        "This is a hardware limitation, not a sign you need Hugging Face PRO."
    )
result = output.images[0]
RESULT_PATH = ROOT / ("manisha_tryon_" + STYLE_CODE + "_research.png")
result.save(RESULT_PATH)
print("Internal evaluation result saved:", RESULT_PATH)
display(result)

ModuleNotFoundError: No module named 'fashn_vton'

## 7 · Compare the originals and generated preview

The output is a synthetic illustration, **not a size/fit guarantee**. Patterns, garment lengths and body details may be inaccurate.

In [ ]:
from IPython.display import display
print("Original catalogue garment:")
display(garment)
print("Official demo person:")
display(person)
print("AI-generated internal evaluation:")
display(result)

## 8 · Download your result (optional) and clear your temporary file

Use the output only to assess the model internally. Do **not** put generated person images on the public Manisha Overseas website. After evaluating, choose **Runtime → Disconnect and delete runtime** to remove the temporary VM; Google may still have separate operational retention.

In [ ]:
# Optional: uncomment only if you need a local copy for internal assessment.
# from google.colab import files
# files.download(str(RESULT_PATH))

print("Local result:", RESULT_PATH)
print("To remove the local result now, run:")
print("RESULT_PATH.unlink(missing_ok=True)")
print("Then use Runtime → Disconnect and delete runtime to discard the temporary VM.")